Le rôle de ce notebook est de générer des données de fraudes à partir du DAG causal généré

In [1]:
import pandas as pd
data = pd.read_csv("../Fraud Detection Dataset.csv")
data.head()


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
0,T1,4174,1292.76,ATM Withdrawal,16.0,Tablet,San Francisco,0,119,13,Debit Card,0
1,T2,4507,1554.58,ATM Withdrawal,13.0,Mobile,New York,4,79,3,Credit Card,0
2,T3,1860,2395.02,ATM Withdrawal,NaN,Mobile,NaN,3,115,9,NaN,0
3,T4,2294,100.10,Bill Payment,15.0,Desktop,Chicago,4,3,4,UPI,0
4,T5,2130,1490.50,POS Payment,19.0,Mobile,San Francisco,2,57,7,Credit Card,0


In [15]:
import os
import json
import re
import time
import urllib3
import requests

# ─────────────────────────────────────────────
# 1. CONFIGURATION API
# ─────────────────────────────────────────────

OLLAMA_API_URL = "https://ollama-api.lab.groupe-genes.fr/api/generate"
MODEL_NAME = "mistral"

SKIP_SSL_VERIFY = True
MAX_RETRIES_LOAD = 2
RETRY_DELAY_SECONDS = 2

if SKIP_SSL_VERIFY:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ─────────────────────────────────────────────
# 2. CHARGEMENT ET FORMATAGE DU GRAPHE CAUSAL
# ─────────────────────────────────────────────

def _normalize_edges(edges_raw: list) -> list:
    FROM_KEYS = ["from", "de", "source", "cause",  "parent", "src", "origine"]
    TO_KEYS   = ["to",   "vers", "target", "cible", "effet",  "child", "dst"]
    JUST_KEYS = ["justification", "justification_experte", "raison", "label", "reason"]

    normalized = []
    for e in edges_raw:
        if isinstance(e, dict):
            src = next((e[k] for k in FROM_KEYS if k in e), None)
            tgt = next((e[k] for k in TO_KEYS   if k in e), None)
            jst = next((e[k] for k in JUST_KEYS if k in e), "")
            if src and tgt:
                normalized.append({
                    "from":          src,
                    "to":            tgt,
                    "justification": jst,
                    # on conserve aussi les méta-données de confiance si présentes
                    "confiance":     e.get("confiance_pct", e.get("confiance", ""))
                })
        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            normalized.append({"from": e[0], "to": e[1], "justification": ""})

    return normalized

def load_causal_graph(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    NODE_KEYS = ["nodes", "noeuds", "nœuds", "vertices", "variables", "features", "columns"]
    EDGE_KEYS = ["edges", "liens", "links", "causal_links", "arcs", "arrows", "relations"]

    node_key = next((k for k in NODE_KEYS if k in raw), None)
    edge_key  = next((k for k in EDGE_KEYS if k in raw), None)

    if node_key and edge_key:
        edges = _normalize_edges(raw[edge_key])
        return {
            "nodes":      raw[node_key],
            "edges":      edges,
            "commentaire": raw.get("commentaire_expert", "")
        }

    raise ValueError(
        f"Format non reconnu. Clés trouvées : {list(raw.keys())}"
    )


def format_graph_for_prompt(graph: dict) -> str:
    lines = ["Variables du dataset :", ""]
    for node in graph["nodes"]:
        lines.append(f"  - {node}")

    lines += ["", "Relations causales (A → B : justification [confiance]) :", ""]
    for edge in graph["edges"]:
        confiance = f" [confiance : {edge['confiance']}]" if edge.get("confiance") else ""
        lines.append(
            f"  - {edge['from']} → {edge['to']} : {edge['justification']}{confiance}"
        )

    # Nœuds puits = variable(s) cible(s)
    sources = {e["from"] for e in graph["edges"]}
    targets = {e["to"]   for e in graph["edges"]}
    sink_nodes = targets - sources
    if sink_nodes:
        lines += ["", f"Variable(s) cible(s) : {', '.join(sink_nodes)}"]

    # Commentaire expert si présent
    if graph.get("commentaire"):
        lines += ["", f"Contexte expert : {graph['commentaire'][:300]}"]

    return "\n".join(lines)


def get_feature_list(graph: dict) -> list[str]:
    """Retourne la liste des variables hors cibles (features à générer)."""
    targets = {e["to"] for e in graph["edges"]} - {e["from"] for e in graph["edges"]}
    return [n for n in graph["nodes"] if n not in targets]


# ─────────────────────────────────────────────
# 3. CONSTRUCTION DU PROMPT
# ─────────────────────────────────────────────

def build_prompt(graph: dict, n_samples: int = 5) -> str:
    graph_description = format_graph_for_prompt(graph)
    features = get_feature_list(graph)
    fields_str = ", ".join(features + ["Fraudulent"])

    return f"""
Tu es un expert en détection de fraude bancaire.

Le graphe causal suivant représente les relations causales entre les variables
du dataset de détection de fraude. Chaque flèche indique qu'une variable influence
directement une autre.

{graph_description}

Ta tâche est de GÉNÉRER des cas de fraude réalistes en respectant scrupuleusement
ces relations causales :
- Les valeurs générées doivent être cohérentes avec les dépendances causales décrites
- Chaque transaction doit contenir des signaux suspects typiques d'une fraude
- Respecte les chemins causaux (ex: si Previous_Fraudulent_Transactions est élevé,
  alors Number_of_Transactions_Last_24H et Transaction_Amount doivent l'être aussi)
- Raisonne étape par étape mais ne montre PAS ton raisonnement
- Réponds UNIQUEMENT avec un objet JSON valide, sans texte autour, sans markdown

Champs attendus pour chaque transaction : {fields_str}

Voici des exemples de transactions FRAUDULEUSES :

Exemple 1 :
{{
  "User_ID": "USR_8821",
  "Account_Age": 1,
  "Previous_Fraudulent_Transactions": 3,
  "Number_of_Transactions_Last_24H": 12,
  "Transaction_Amount": 2450,
  "Transaction_Type": "Bill Payment",
  "Device_Used": "Tablet",
  "Location": "Boston",
  "Time_of_Transaction": 2,
  "Fraudulent": 1
}}

Exemple 2 :
{{
  "User_ID": "USR_3345",
  "Account_Age": 2,
  "Previous_Fraudulent_Transactions": 5,
  "Number_of_Transactions_Last_24H": 9,
  "Transaction_Amount": 1800,
  "Transaction_Type": "Online Transfer",
  "Device_Used": "Tablet",
  "Location": "Boston",
  "Time_of_Transaction": 23,
  "Fraudulent": 1
}}

Maintenant, génère {n_samples} nouvelles transactions frauduleuses réalistes.
Réponds avec un JSON valide contenant une clé "transactions" qui est une liste
de {n_samples} objets ayant exactement les mêmes champs que les exemples.
"""


# ─────────────────────────────────────────────
# 4. APPEL API OLLAMA AVEC RETRY
# ─────────────────────────────────────────────

def call_ollama(prompt: str) -> str | None:
    """Appelle l'API Ollama et retourne le texte généré."""
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.5,
            "num_predict": 1200,
        }
    }

    for attempt in range(1, MAX_RETRIES_LOAD + 1):
        print(f"🔄 Tentative {attempt}/{MAX_RETRIES_LOAD} — appel à {MODEL_NAME}...")
        try:
            response = requests.post(
                OLLAMA_API_URL,
                json=payload,
                verify=not SKIP_SSL_VERIFY,
                timeout=120
            )
            response.raise_for_status()
            return response.json().get("response", "")

        except requests.exceptions.HTTPError as e:
            if response.status_code in (503, 429):
                print(f"⏳ Modèle en chargement (HTTP {response.status_code}), "
                      f"nouvel essai dans {RETRY_DELAY_SECONDS}s...")
                time.sleep(RETRY_DELAY_SECONDS)
            else:
                print(f"❌ Erreur HTTP : {e}")
                break

        except requests.exceptions.RequestException as e:
            print(f"❌ Erreur réseau : {e}")
            break

    return None


# ─────────────────────────────────────────────
# 5. EXTRACTION ET PARSING DU JSON
# ─────────────────────────────────────────────

def extract_json(text: str) -> dict | list | None:
    """
    Tente plusieurs stratégies pour extraire un JSON valide
    depuis la réponse brute du modèle.
    """
    # Stratégie 1 : bloc ```json ... ```
    match = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    # Stratégie 2 : premier objet { ... } ou tableau [ ... ] trouvé
    for start_char, end_char in [('{', '}'), ('[', ']')]:
        start = text.find(start_char)
        end = text.rfind(end_char) + 1
        if start != -1 and end > start:
            try:
                return json.loads(text[start:end])
            except json.JSONDecodeError:
                pass

    return None


# ─────────────────────────────────────────────
# 6. VALIDATION DES TRANSACTIONS GÉNÉRÉES
# ─────────────────────────────────────────────

def validate_transactions(transactions: list) -> list:
    """
    Vérifie uniquement que Fraudulent vaut 1.
    Les champs générés par le LLM sont acceptés tels quels.
    """
    valid, invalid = [], []

    for i, tx in enumerate(transactions):
        fraudulent_val = tx.get("Fraudulent")
        if fraudulent_val not in (1, "1", True):
            print(f"  ⚠️  Transaction {i+1} — Fraudulent != 1 "
                  f"(valeur={fraudulent_val!r}, type={type(fraudulent_val).__name__})")
            invalid.append(tx)
            continue

        tx["Fraudulent"] = 1  # normalisation str → int
        valid.append(tx)

    return valid

# ─────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":
    N_SAMPLES = 5

    # Chargement du graphe
    graph = load_causal_graph("dag_causal_fraude.json")
    print(f"   {len(graph['nodes'])} nœuds, {len(graph['edges'])} arêtes chargés.\n")

    # Affichage de la description générée
    print("=== Description du graphe injectée dans le prompt ===")
    print(format_graph_for_prompt(graph))
    print()

    # Construction et envoi du prompt
    prompt = build_prompt(graph, n_samples=N_SAMPLES)
    print(f"🚀 Génération de {N_SAMPLES} transactions frauduleuses avec {MODEL_NAME}\n", flush=True)

    raw_text = call_ollama(prompt)
    if raw_text is None:
        print("❌ Impossible d'obtenir une réponse du modèle.")
        exit(1)

    print("✅ Réponse reçue\n")
    print("=== Réponse brute ===")
    print(raw_text, flush=True)

    # Parsing
    parsed = extract_json(raw_text)
    if parsed is None:
        print("\n⚠️  Aucun JSON valide trouvé dans la réponse.")
        exit(1)

    transactions = parsed if isinstance(parsed, list) else parsed.get("transactions", [])

    # Validation
    print(f"\n=== Validation des {len(transactions)} transaction(s) ===", flush=True)
    valid_transactions = validate_transactions(transactions)
    print(f"   ✅ {len(valid_transactions)} valide(s) / {len(transactions)} générée(s)\n", flush=True)

    # Affichage final
    print("=== Transactions valides ===")
    for i, tx in enumerate(valid_transactions, 1):
        print(f"\n  Transaction {i} :")
        for k, v in tx.items():
            print(f"    {k}: {v}")

    # Sauvegarde optionnelle en JSON
    output_path = "transactions_frauduleuses.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump({"transactions": valid_transactions}, f, indent=2, ensure_ascii=False)
    print(f"\n💾 Résultats sauvegardés dans : {output_path}")

   8 nœuds, 11 arêtes chargés.

=== Description du graphe injectée dans le prompt ===
Variables du dataset :

  - Account_Age
  - Location
  - Transaction_Type
  - Device_Used
  - Number_of_Transactions_Last_24H
  - Payment_Method
  - Time_of_Transaction
  - Fraudulent

Relations causales (A → B : justification [confiance]) :

  - Location → Fraudulent : Some locations have higher rates of fraud [confiance : 34.6%]
  - Transaction_Type → Fraudulent : Online purchases, POS payments, bill payments, bank transfers, and ATM withdrawals can potentially involve fraud [confiance : 12.7%]
  - Device_Used → Fraudulent : Certain devices might be more susceptible to fraud [confiance : 12.7%]
  - Payment_Method → Fraudulent : Different payment methods may involve varying levels of risk for fraud [confiance : 12.7%]
  - Location → Transaction_Type : Different locations might prefer different transaction types [confiance : 7.7%]
  - Location → Device_Used : Different devices might be preferred in ce

🔄 Tentative 1/2 — appel à mistral...
✅ Réponse reçue

=== Réponse brute ===
 {
  "transactions": [
    {
      "User_ID": "USR_7719",
      "Account_Age": 3,
      "Previous_Fraudulent_Transactions": 4,
      "Number_of_Transactions_Last_24H": 15,
      "Transaction_Amount": 2800,
      "Transaction_Type": "Online Purchase",
      "Device_Used": "Smartphone",
      "Location": "Los Angeles",
      "Time_of_Transaction": 19,
      "Fraudulent": 1
    },
    {
      "User_ID": "USR_2204",
      "Account_Age": 6,
      "Previous_Fraudulent_Transactions": 7,
      "Number_of_Transactions_Last_24H": 18,
      "Transaction_Amount": 3200,
      "Transaction_Type": "ATM Withdrawal",
      "Device_Used": "Laptop",
      "Location": "Chicago",
      "Time_of_Transaction": 6,
      "Fraudulent": 1
    },
    {
      "User_ID": "USR_4432",
      "Account_Age": 5,
      "Previous_Fraudulent_Transactions": 2,
      "Number_of_Transactions_Last_24H": 10,
      "Transaction_Amount": 2000,
      "Trans

Nous allons maintenant évaluer la qualité des données synthétiques générées comme on a fait précédemment

In [17]:
!pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 18.6 MB/s  0:00:00 eta 0:00:01


In [19]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_curve, auc, classification_report,
    precision_score, recall_score, f1_score, accuracy_score
)
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════
# 1. CHARGEMENT DES DONNÉES
# ═══════════════════════════════════════════════════════════

def load_real_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    print(f"✅ Données réelles chargées : {df.shape}")
    print(df["Fraudulent"].value_counts().to_string())
    return df


def load_synthetic_data(json_path: str) -> pd.DataFrame:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data["transactions"])
    print(f"\n✅ Données synthétiques chargées : {df.shape}")
    print(df["Fraudulent"].value_counts().to_string())
    return df


# ═══════════════════════════════════════════════════════════
# 2. PREPROCESSING (réutilise la logique du notebook)
# ═══════════════════════════════════════════════════════════

def preprocess(df: pd.DataFrame, fit_encoders: dict = None) -> tuple:
    """
    Encode les variables catégorielles et sépare X / y.
    Si fit_encoders est fourni, applique les encodeurs existants
    (cas où on encode les synth. avec les encodeurs du réel).
    Retourne (X, y, encoders).
    """
    df = df.copy()

    # Suppression des colonnes non prédictives
    drop_cols = [c for c in ["User_ID", "Transaction_ID"] if c in df.columns]
    df.drop(columns=drop_cols, inplace=True, errors="ignore")

    # Valeurs manquantes : remplacées par la moyenne / mode
    for col in df.select_dtypes(include=["float64", "int64"]).columns:
        if col != "Fraudulent":
            df[col].fillna(df[col].mean(), inplace=True)
    for col in df.select_dtypes(include=["object", "category"]).columns:
        df[col].fillna(df[col].mode()[0], inplace=True)

    y = df["Fraudulent"].astype(int)
    X = df.drop(columns=["Fraudulent"])

    encoders = fit_encoders if fit_encoders else {}
    for col in X.select_dtypes(include=["object", "category"]).columns:
        if col in encoders:
            # Gestion des catégories inconnues dans les synth.
            le = encoders[col]
            X[col] = X[col].apply(
                lambda v: le.transform([v])[0] if v in le.classes_ else -1
            )
        else:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            encoders[col] = le

    # Alignement des colonnes (synth. peut avoir des colonnes différentes)
    return X, y, encoders


def align_columns(X_source: pd.DataFrame, X_target: pd.DataFrame) -> pd.DataFrame:
    """Aligne les colonnes de X_source sur X_target (remplissage à 0 si manquant)."""
    for col in X_target.columns:
        if col not in X_source.columns:
            X_source[col] = 0
    return X_source[X_target.columns]


# ═══════════════════════════════════════════════════════════
# 3. MODÈLES (repris du notebook)
# ═══════════════════════════════════════════════════════════

MODELS = {
    "RandomForest":   RandomForestClassifier(n_estimators=100, random_state=42),
    "LogisticReg":    LogisticRegression(max_iter=1000, random_state=42),
    "SVM":            SVC(kernel="rbf", C=5, gamma="scale",
                          probability=True, random_state=42),
    "LightGBM":       lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5,
                                          random_state=42, verbose=-1),
}


def evaluate_model(model, X_train, y_train, X_test, y_test) -> dict:
    """Entraîne et évalue un modèle, retourne toutes les métriques du notebook."""
    model.fit(X_train, y_train)
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    auc_score = auc(fpr, tpr)

    # Indice de Youden (seuil optimal)
    youden      = tpr - fpr
    best_idx    = np.argmax(youden)
    best_thresh = thresholds[best_idx]
    y_pred_opt  = (y_pred_proba >= best_thresh).astype(int)

    tn = np.sum((y_pred_opt == 0) & (y_test == 0))
    fp = np.sum((y_pred_opt == 1) & (y_test == 0))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {
        "auc":         round(auc_score, 4),
        "accuracy":    round(accuracy_score(y_test, y_pred), 4),
        "recall":      round(recall_score(y_test, y_pred_opt, zero_division=0), 4),
        "specificity": round(specificity, 4),
        "precision":   round(precision_score(y_test, y_pred_opt, zero_division=0), 4),
        "f1":          round(f1_score(y_test, y_pred_opt, zero_division=0), 4),
        "youden":      round(float(youden[best_idx]), 4),
        "threshold":   round(float(best_thresh), 4),
        "fpr": fpr, "tpr": tpr,  # pour les courbes ROC
    }


# ═══════════════════════════════════════════════════════════
# 4. TROIS PROTOCOLES D'ENTRAÎNEMENT
# ═══════════════════════════════════════════════════════════
#
#  A) TRTR — Train Real  / Test Real  (baseline du notebook)
#  B) TSTR — Train Synth / Test Real  (qualité des synth.)
#  C) TSAR — Train Synth+Real / Test Real (apport des synth.)
#
# ═══════════════════════════════════════════════════════════

def run_experiments(
    df_real: pd.DataFrame,
    df_synth: pd.DataFrame,
    test_size: float = 0.2,
) -> dict:

    # ── Preprocessing ──────────────────────────────────────
    X_real, y_real, encoders = preprocess(df_real)
    X_synth, y_synth, _      = preprocess(df_synth, fit_encoders=encoders)

    # Alignement des colonnes synth → réel
    X_synth = align_columns(X_synth, X_real)

    # Split stratifié sur les données réelles
    X_train_r, X_test, y_train_r, y_test = train_test_split(
        X_real, y_real, test_size=test_size, random_state=42, stratify=y_real
    )

    # Données augmentées : train réel + tout le synthétique
    X_train_aug = pd.concat([X_train_r, X_synth], ignore_index=True)
    y_train_aug  = pd.concat([y_train_r, y_synth], ignore_index=True)

    results = {}

    for model_name, model_proto in MODELS.items():
        print(f"\n  📊 {model_name}...", flush=True)
        results[model_name] = {}

        # A) TRTR
        from sklearn.base import clone
        results[model_name]["TRTR"] = evaluate_model(
            clone(model_proto), X_train_r, y_train_r, X_test, y_test
        )

        # B) TSTR
        results[model_name]["TSTR"] = evaluate_model(
            clone(model_proto), X_synth, y_synth, X_test, y_test
        )

        # C) TSAR
        results[model_name]["TSAR"] = evaluate_model(
            clone(model_proto), X_train_aug, y_train_aug, X_test, y_test
        )

    return results, X_test, y_test


# ═══════════════════════════════════════════════════════════
# 5. VISUALISATION (style notebook)
# ═══════════════════════════════════════════════════════════

PROTOCOL_COLORS = {
    "TRTR": "#2196F3",   # bleu  — baseline réel
    "TSTR": "#E84545",   # rouge — synthétique pur
    "TSAR": "#4CAF50",   # vert  — augmenté
}
PROTOCOL_LABELS = {
    "TRTR": "Train Réel / Test Réel (baseline)",
    "TSTR": "Train Synth / Test Réel",
    "TSAR": "Train Synth+Réel / Test Réel",
}

def plot_roc_curves(results: dict):
    """Courbes ROC comparatives — une sous-figure par modèle."""
    n = len(results)
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    axes = axes.flatten()

    for ax, (model_name, protocols) in zip(axes, results.items()):
        for proto, metrics in protocols.items():
            ax.plot(
                metrics["fpr"], metrics["tpr"],
                color=PROTOCOL_COLORS[proto],
                label=f"{PROTOCOL_LABELS[proto]} (AUC={metrics['auc']:.3f})",
                linewidth=2
            )
        ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5)
        ax.set_title(model_name, fontsize=12, fontweight="bold")
        ax.set_xlabel("Taux de Faux Positifs")
        ax.set_ylabel("Taux de Vrais Positifs")
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(alpha=0.3)

    plt.suptitle("Courbes ROC — Comparaison TRTR / TSTR / TSAR",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("roc_curves_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 Sauvegardé : roc_curves_comparison.png")


def build_summary_table(results: dict) -> pd.DataFrame:
    """
    Tableau récapitulatif au même format multi-index que le notebook.
    """
    metrics_to_show = ["auc", "recall", "specificity", "precision", "f1", "youden"]

    index   = pd.Index(results.keys(), name="Modèle")
    columns = pd.MultiIndex.from_tuples(
        [(proto, m)
         for proto in ["TRTR", "TSTR", "TSAR"]
         for m in metrics_to_show],
        names=["Protocole", "Métrique"]
    )

    rows = []
    for model_name, protocols in results.items():
        row = []
        for proto in ["TRTR", "TSTR", "TSAR"]:
            for m in metrics_to_show:
                row.append(protocols[proto].get(m, float("nan")))
        rows.append(row)

    return pd.DataFrame(rows, index=index, columns=columns)


def plot_metric_bars(summary: pd.DataFrame, metric: str = "auc"):
    """Barres comparatives pour une métrique donnée."""
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(summary.index))
    width = 0.25

    for i, proto in enumerate(["TRTR", "TSTR", "TSAR"]):
        vals = summary[proto][metric].values
        bars = ax.bar(x + i * width, vals, width,
                      label=PROTOCOL_LABELS[proto],
                      color=PROTOCOL_COLORS[proto], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x + width)
    ax.set_xticklabels(summary.index, fontsize=10)
    ax.set_ylabel(metric.upper())
    ax.set_title(f"Comparaison {metric.upper()} — TRTR vs TSTR vs TSAR",
                 fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"comparison_{metric}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"💾 Sauvegardé : comparison_{metric}.png")


def tstr_score(results: dict) -> float:
    """
    Score de qualité des données synthétiques :
    ratio moyen AUC_TSTR / AUC_TRTR sur tous les modèles.
    Un score proche de 1.0 indique que les synth. sont aussi
    informatifs que les données réelles.
    """
    ratios = [
        v["TSTR"]["auc"] / v["TRTR"]["auc"]
        for v in results.values()
        if v["TRTR"]["auc"] > 0
    ]
    return round(np.mean(ratios), 4)


# ═══════════════════════════════════════════════════════════
# 6. MAIN
# ═══════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Chargement ──────────────────────────────────────────
    df_real  = load_real_data("../Fraud Detection Dataset.csv")
    df_synth = load_synthetic_data("transactions_frauduleuses.json")

    # ── Expériences ─────────────────────────────────────────
    print("\n🚀 Lancement des expériences (TRTR / TSTR / TSAR)...")
    results, X_test, y_test = run_experiments(df_real, df_synth)

    # ── Tableau récapitulatif ────────────────────────────────
    summary = build_summary_table(results)
    print("\n=== Tableau récapitulatif ===")
    print(summary.to_string())

    # ── Score TSTR global ────────────────────────────────────
    score = tstr_score(results)
    print(f"\n📈 Score TSTR global (AUC_TSTR / AUC_TRTR) : {score:.4f}")
    if score >= 0.90:
        print("   ✅ Excellent — les données synthétiques sont très proches du réel")
    elif score >= 0.75:
        print("   ✅ Bon — les données capturent bien la structure")
    elif score >= 0.60:
        print("   ⚠️  Acceptable — à améliorer (plus d'échantillons, meilleur prompt)")
    else:
        print("   ❌ Insuffisant — les synth. n'apportent pas d'information utile")

    # ── Visualisations ───────────────────────────────────────
    plot_roc_curves(results)
    plot_metric_bars(summary, metric="auc")
    plot_metric_bars(summary, metric="recall")

    # ── Sauvegarde JSON ──────────────────────────────────────
    exportable = {
        model: {
            proto: {k: v for k, v in m.items() if k not in ("fpr", "tpr")}
            for proto, m in protocols.items()
        }
        for model, protocols in results.items()
    }
    exportable["tstr_score"] = score
    with open("resultats_tstr.json", "w", encoding="utf-8") as f:
        json.dump(exportable, f, indent=2, ensure_ascii=False)
    print("\n💾 Résultats sauvegardés dans resultats_tstr.json")

✅ Données réelles chargées : (51000, 12)
Fraudulent
0    48490
1     2510

✅ Données synthétiques chargées : (5, 10)
Fraudulent
1    5

🚀 Lancement des expériences (TRTR / TSTR / TSAR)...

  📊 RandomForest...


IndexError: index 1 is out of bounds for axis 1 with size 1